In [67]:
import psycopg2
import tkinter as tk
from tkinter import messagebox, simpledialog, scrolledtext

In [ ]:
#criação do banco

conn = psycopg2.connect(
    dbname="postgres",       
    user="postgres",         
    password="root",    
    host="localhost",
    port="5432"
)


cursor = conn.cursor()
conn.autocommit = True
cursor.execute("CREATE DATABASE universidades")

cursor.close()
conn.close()

In [ ]:
# Conexão com o banco de dados
db     = "universidades"
user   = "postgres"
senha  = "root"
host   = "localhost"
port   = "5432"

conexao = psycopg2.connect(
    dbname= db,     # nome do banco criado
    user= user,          # meu usuário
    password= senha, # senha do PostgreSQL
    host= host,          
    port= port                #  porta padrão
)
cursor = conexao.cursor()
conexao.autocommit = True

cursor.execute("""
CREATE TABLE IF NOT EXISTS alunos (
    id SERIAL PRIMARY KEY,
    nome VARCHAR(100) NOT NULL,
    curso VARCHAR(100) NOT NULL
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS notas (
    id SERIAL PRIMARY KEY,
    aluno_id INTEGER REFERENCES alunos(id) ON DELETE CASCADE,
    nota NUMERIC(4,2) NOT NULL
);
""")

cursor.close()
conexao.close()



In [82]:
def conectar():
    try:
        conn = psycopg2.connect(
            dbname=db,
            user=user,
            password=senha,
            host=host,
            port=port
        )
        return conn
    except Exception as e:
        print("Erro de conexão:", e)
        return None

# ——— CRUD ———
def inserir_aluno(nome, curso):
    conn = conectar()
    if not conn:
        return False
    try:
        with conn:
            with conn.cursor() as cursor:
                cursor.execute(
                    "INSERT INTO alunos (nome, curso) VALUES (%s, %s)",
                    (nome, curso)
                )
        messagebox.showinfo("Sucesso", f"Aluno '{nome}' inserido.")
        return True
    except Exception as e:
        messagebox.showerror("Erro ao inserir aluno", str(e))
        return False
    finally:
        conn.close()


def inserir_nota(aluno_id, nota):
    conn = conectar()
    if not conn:
        return False
    try:
        with conn:
            with conn.cursor() as cursor:
                cursor.execute(
                    "INSERT INTO notas (aluno_id, nota) VALUES (%s, %s)",
                    (aluno_id, nota)
                )
        messagebox.showinfo("Sucesso", f"Nota {nota:.2f} inserida para aluno {aluno_id}.")
        return True
    except Exception as e:
        messagebox.showerror("Erro ao inserir nota", str(e))
        return False
    finally:
        conn.close()


def listar_alunos():
    conn = conectar()
    if not conn:
        return []
    try:
        with conn.cursor() as cursor:
            cursor.execute("SELECT id, nome, curso FROM alunos ORDER BY id;")
            return cursor.fetchall()
    except Exception as e:
        messagebox.showerror("Erro ao listar alunos", str(e))
        return []
    finally:
        conn.close()


def listar_notas():
    conn = conectar()
    if not conn:
        return []
    try:
        with conn.cursor() as cursor:
            cursor.execute("""
                SELECT n.id, a.id, a.nome, a.curso, n.nota
                FROM notas n
                JOIN alunos a ON a.id = n.aluno_id
                ORDER BY n.id;
            """)
            return cursor.fetchall()
    except Exception as e:
        messagebox.showerror("Erro ao listar notas", str(e))
        return []
    finally:
        conn.close()


def atualizar_nota(nota_id, nova_nota):
    conn = conectar()
    if not conn:
        return False
    try:
        with conn:
            with conn.cursor() as cursor:
                cursor.execute(
                    "UPDATE notas SET nota = %s WHERE id = %s",
                    (nova_nota, nota_id)
                )
        messagebox.showinfo("Sucesso", f"Nota {nota_id} atualizada para {nova_nota:.2f}.")
        return True
    except Exception as e:
        messagebox.showerror("Erro ao atualizar nota", str(e))
        return False
    finally:
        conn.close()


def deletar_aluno(aluno_id):
    conn = conectar()
    if not conn:
        return False
    try:
        with conn:
            with conn.cursor() as cursor:
                cursor.execute(
                    "DELETE FROM alunos WHERE id = %s",
                    (aluno_id,)
                )
        messagebox.showinfo("Sucesso", f"Aluno {aluno_id} deletado.")
        return True
    except Exception as e:
        messagebox.showerror("Erro ao deletar aluno", str(e))
        return False
    finally:
        conn.close()


def buscar_aluno_por_id(id_aluno):
    conn = conectar()
    if not conn:
        return []
    try:
        with conn.cursor() as cursor:
            cursor.execute("""
                SELECT a.id, a.nome, a.curso, n.nota
                FROM alunos a
                LEFT JOIN notas n ON a.id = n.aluno_id
                WHERE a.id = %s;
            """, (id_aluno,))
            return cursor.fetchall()
    except Exception as e:
        messagebox.showerror("Erro ao buscar aluno", str(e))
        return []
    finally:
        conn.close()

In [83]:
# ——— Interface Tkinter ———
class App:
    def __init__(self, master):
        self.master = master
        master.title("Sistema de Alunos e Notas")
        master.geometry("600x450")

        # Botões de ação
        actions = [
            ("Inserir Aluno",       self.gui_inserir_aluno),
            ("Inserir Nota",        self.gui_inserir_nota),
            ("Listar Alunos",       self.gui_listar_alunos),
            ("Listar Notas",        self.gui_listar_notas),
            ("Atualizar Nota",      self.gui_atualizar_nota),
            ("Deletar Aluno",       self.gui_deletar_aluno),
            ("Buscar Aluno por ID", self.gui_buscar_aluno),
        ]
        for text, cmd in actions:
            tk.Button(master, text=text, width=25, command=cmd).pack(pady=5)

        # Área de texto para resultados
        self.texto = scrolledtext.ScrolledText(master, width=80, height=15)
        self.texto.pack(pady=10)

    def gui_inserir_aluno(self):
        nome  = simpledialog.askstring("Nome", "Digite o nome do aluno:")
        curso = simpledialog.askstring("Curso", "Digite o curso do aluno:")
        if nome and curso:
            inserir_aluno(nome, curso)
            self.gui_listar_alunos()

    def gui_inserir_nota(self):
        aluno_id = simpledialog.askinteger("Aluno ID", "Digite o ID do aluno:")
        nota     = simpledialog.askfloat("Nota", "Digite a nota (ex: 8.50):")
        if aluno_id and nota is not None:
            inserir_nota(aluno_id, nota)
            self.gui_listar_notas()

    def gui_listar_alunos(self):
        self.texto.delete(1.0, tk.END)
        alunos = listar_alunos()
        if not alunos:
            self.texto.insert(tk.END, "Nenhum aluno cadastrado.\n")
        for id_a, nome, curso in alunos:
            self.texto.insert(
                tk.END,
                f"ID: {id_a} | Nome: {nome} | Curso: {curso}\n"
            )

    def gui_listar_notas(self):
        self.texto.delete(1.0, tk.END)
        notas = listar_notas()
        if not notas:
            self.texto.insert(tk.END, "Nenhuma nota cadastrada.\n")
            return
        for nota_id, aluno_id, nome, curso, valor in notas:
            self.texto.insert(
                tk.END,
                f"Nota ID: {nota_id} | Aluno ID: {aluno_id} | Nome: {nome} | Curso: {curso} | Nota: {valor:.2f}\n"
            )

    def gui_atualizar_nota(self):
        nota_id   = simpledialog.askinteger("Nota ID", "Digite o ID da nota:")
        nova_nota = simpledialog.askfloat("Nova Nota", "Digite a nova nota:")
        if nota_id and nova_nota is not None:
            atualizar_nota(nota_id, nova_nota)
            self.gui_listar_notas()

    def gui_deletar_aluno(self):
        aluno_id = simpledialog.askinteger("Aluno ID", "Digite o ID do aluno para deletar:")
        if aluno_id:
            deletar_aluno(aluno_id)
            self.gui_listar_alunos()

    def gui_buscar_aluno(self):
        id_aluno = simpledialog.askinteger("Buscar Aluno", "Digite o ID do aluno:")
        if id_aluno:
            resultados = buscar_aluno_por_id(id_aluno)
            self.texto.delete(1.0, tk.END)
            if resultados:
                for id_a, nome, curso, nota in resultados:
                    self.texto.insert(
                        tk.END,
                        f"ID: {id_a} | Nome: {nome} | Curso: {curso} | Nota: {nota if nota is not None else 'Sem nota'}\n"
                    )
            else:
                self.texto.insert(tk.END, f"Nenhum aluno com ID {id_aluno}.\n")

if __name__ == "__main__":
    criar_tabelas()
    root = tk.Tk()
    app = App(root)
    root.mainloop()
